# Constrained Optimization & KKT

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/optimization-ml/03-constrained-optimization

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'
plt.rcParams['grid.color'] = '#2e3347'

BRAND  = '#818cf8'
TEAL   = '#14b8a6'
YELLOW = '#f59e0b'
ROSE   = '#f43f5e'

rng = np.random.default_rng(42)

## Intuition — optimizing under limits

Real problems come with **constraints**: minimize cost *subject to* a budget, maximize margin
*subject to* every point classified correctly. Two tools handle this. For **equality**
constraints, **Lagrange multipliers** say the optimum sits where the objective's gradient is
parallel to the constraint's (`∇f = λ∇h`) — you can't improve without violating the constraint.
For **inequality** constraints, the **KKT conditions** generalize this with four requirements
(stationarity, primal & dual feasibility, complementary slackness). These aren't abstract:
they're exactly what defines an **SVM's** support vectors — the few points whose constraints are
*active*. We verify all of it numerically.

## 1 — Equality Constraint Visualisation

**Problem:** minimise $f(x,y) = x^2 + y^2$ subject to $x + y = 1$.

The contour lines of $f$ are circles centred at the origin.  
The constraint is the diagonal line $x + y = 1$.  
The optimum is where the **smallest circle is tangent to the line** — i.e., $\nabla f \parallel \nabla h$.

At the tangency point $(0.5, 0.5)$:
- $\nabla f = (2x, 2y) = (1, 1)$
- $\nabla h = (1, 1)$

They are parallel ✓, confirming the Lagrangian stationarity condition.

In [ ]:
xy = np.linspace(-0.5, 1.5, 300)
X2d, Y2d = np.meshgrid(xy, xy)
F = X2d**2 + Y2d**2

fig, ax = plt.subplots(figsize=(7, 6))
levels = np.array([0.05, 0.1, 0.2, 0.3, 0.5, 0.75, 1.0, 1.5])
cs = ax.contour(X2d, Y2d, F, levels=levels, colors=BRAND, alpha=0.6, linewidths=1.2)
ax.clabel(cs, fmt='%.2f', colors='#94a3b8', fontsize=8)

# Constraint line x + y = 1
cx = np.linspace(-0.3, 1.3, 200)
ax.plot(cx, 1 - cx, color=YELLOW, linewidth=2, label='constraint $x+y=1$')

# Optimum
ax.scatter([0.5], [0.5], color=TEAL, s=100, zorder=6, label='optimum $(0.5, 0.5)$')

# Gradient arrows at optimum
scale = 0.18
ax.annotate('', xy=(0.5 + scale, 0.5 + scale), xytext=(0.5, 0.5),
            arrowprops=dict(arrowstyle='->', color=TEAL, lw=2))
ax.text(0.5 + scale + 0.02, 0.5 + scale + 0.02, r'$\nabla f$', color=TEAL, fontsize=11)

ax.annotate('', xy=(0.5 + scale, 0.5 + scale), xytext=(0.5, 0.5),
            arrowprops=dict(arrowstyle='->', color=ROSE, lw=2, linestyle='dashed'))
ax.text(0.5 + scale - 0.18, 0.5 + scale + 0.05, r'$\nabla h$ (parallel!)', color=ROSE, fontsize=9)

ax.set_xlim(-0.4, 1.4)
ax.set_ylim(-0.3, 1.3)
ax.set_xlabel('$x$')
ax.set_ylabel('$y$')
ax.set_title('Lagrange multiplier: $\\nabla f \\parallel \\nabla h$ at the constrained optimum', color='white')
ax.legend()
ax.grid(True, alpha=0.25)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

**What to notice:** the optimum is where the smallest circle just *touches* the constraint
line — tangency. There, `∇f = (1,1)` points the same way as `∇h = (1,1)`, so `∇f = λ∇h` with
`λ=1`. That parallel-gradients condition is the whole idea of Lagrange multipliers: at the
constrained optimum, the objective can only be improved by leaving the constraint.

## 2. The library way — `scipy.optimize.minimize` with a constraint

You don't solve for the tangency by hand; `scipy.optimize.minimize` takes the constraint
directly and returns the constrained optimum. The cell solves the same `min x²+y²  s.t. x+y=1`
and asserts it lands on `(0.5, 0.5)`.

In [ ]:
from scipy.optimize import minimize

res = minimize(lambda v: v[0]**2 + v[1]**2, x0=np.array([0.0, 0.0]),
               constraints={'type': 'eq', 'fun': lambda v: v[0] + v[1] - 1.0})
print('constrained optimum:', res.x.round(4), ' objective:', round(res.fun, 4))
assert np.allclose(res.x, [0.5, 0.5], atol=1e-4), "must find (0.5, 0.5)"
print('scipy constrained solver matches the Lagrange tangency point ✓')

**What to notice:** `scipy` finds `(0.5, 0.5)` — exactly the tangency point the gradient
condition predicted. In practice you hand the constraints to a solver (`scipy`, `cvxpy`), but
knowing `∇f = λ∇h` tells you *why* the answer sits where it does.

## 2 — KKT for a Simple LP

**Minimise** $-2x - y$ subject to $x+y \leq 4$, $x \leq 3$, $y \leq 3$, $x,y \geq 0$.

We draw the feasible polygon and objective contours, then verify the KKT conditions at the optimum.

In [ ]:
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.collections import PatchCollection

# Feasible vertices (corner points of the polygon)
vertices = np.array([
    [0, 0], [3, 0], [3, 1], [1, 3], [0, 3]
])

fig, ax = plt.subplots(figsize=(7, 6))

poly = MplPolygon(vertices, closed=True)
patch = PatchCollection([poly], alpha=0.2, facecolor=BRAND, edgecolor=BRAND, linewidth=2)
ax.add_collection(patch)

# Objective contours -2x - y = c  =>  y = -2x - c
xr = np.linspace(-0.5, 4, 200)
for c in [-2, -4, -6, -7, -8]:
    ax.plot(xr, -c - 2*xr, '--', color=YELLOW, alpha=0.35, linewidth=1)
ax.text(3.5, 0.6, 'objective\ncontours', color=YELLOW, fontsize=8, alpha=0.8)

# Mark all vertices
for v in vertices:
    ax.scatter(*v, color='white', s=40, zorder=5)

# Optimal point
x_opt, y_opt = 3.0, 1.0
ax.scatter([x_opt], [y_opt], color=TEAL, s=120, zorder=7,
           label=f'optimum $(3, 1)$, obj$={-2*x_opt - y_opt}$')

ax.set_xlim(-0.3, 4.2)
ax.set_ylim(-0.3, 4.2)
ax.set_xlabel('$x$')
ax.set_ylabel('$y$')
ax.set_title('LP feasible region + objective contours', color='white')
ax.legend()
ax.grid(True, alpha=0.25)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

# Verify KKT at (3, 1)
print('=== KKT verification at (x*, y*) = (3, 1) ===')
x_s, y_s = 3.0, 1.0
g = np.array([x_s + y_s - 4, x_s - 3, y_s - 3, -x_s, -y_s])
print(f'Constraint values g_i(x*) = {g}  (all <= 0 -> primal feasible ✓)')

# Active constraints: g1 = x+y-4=0, g2 = x-3=0  => mu1, mu2 free; mu3=mu4=mu5=0
# Stationarity: nabla(-2x-y) + mu1*nabla(x+y-4) + mu2*nabla(x-3) = 0
# (-2,-1) + mu1*(1,1) + mu2*(1,0) = (0,0)
# => mu1 + mu2 = 2  and  mu1 = 1  =>  mu1=1, mu2=1
mu = np.array([1.0, 1.0, 0.0, 0.0, 0.0])
print(f'Dual variables mu = {mu}  (all >= 0 -> dual feasible ✓)')
print(f'Complementary slackness mu_i * g_i = {mu * g}  (all == 0 ✓)')

**What to notice:** the LP optimum is a **corner** of the feasible polygon, `(3,1)`, where two
constraints are *active* (`x+y=4` and `x=3`). The printed KKT check confirms all four conditions:
constraints ≤ 0 (feasible), multipliers ≥ 0 (dual feasible), and `μᵢ·gᵢ = 0` for every constraint
(complementary slackness — a multiplier is nonzero only where its constraint is active).

## 3 — SVM Support Vectors via KKT

We generate a 2-class 2-D dataset, solve the **primal SVM** using `scipy.optimize.minimize`,
then plot the margin and highlight which points are support vectors (those where
$y_i(w \cdot x_i + b) = 1$, i.e. the active constraints).

In [ ]:
from sklearn.svm import SVC

# Small linearly-separable dataset
rng2 = np.random.default_rng(5)
X_pos = rng2.standard_normal((10, 2)) + np.array([2.0, 2.0])
X_neg = rng2.standard_normal((10, 2)) + np.array([-2.0, -2.0])
X_svm = np.vstack([X_pos, X_neg])
y_svm = np.hstack([np.ones(10), -np.ones(10)])

# Solve via sklearn (hard margin)
svm = SVC(kernel='linear', C=1e6).fit(X_svm, y_svm)
w_svm = svm.coef_[0]
b_svm = svm.intercept_[0]
margin = 2.0 / np.linalg.norm(w_svm)

# Compute functional margins y_i(w·x_i + b)
func_margins = y_svm * (X_svm @ w_svm + b_svm)
sv_mask = np.abs(func_margins - 1.0) < 0.05   # support vectors: active constraint

print(f'w = {w_svm.round(3)},  b = {b_svm:.3f},  margin = {margin:.3f}')
print(f'Support vector indices: {np.where(sv_mask)[0]}')
print(f'Functional margins (should be >= 1): {func_margins.round(3)}')

# Plot
xx_s = np.linspace(X_svm[:, 0].min() - 1, X_svm[:, 0].max() + 1, 200)
fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(X_pos[:, 0], X_pos[:, 1], c=BRAND, s=50, label='Class +1')
ax.scatter(X_neg[:, 0], X_neg[:, 1], c=ROSE,  s=50, label='Class -1')

# Highlight support vectors
ax.scatter(X_svm[sv_mask, 0], X_svm[sv_mask, 1],
           edgecolors=TEAL, facecolors='none', s=150, linewidths=2,
           label='Support vectors ($\\alpha_i > 0$)')

# Decision boundary and margins
def boundary_y(x_vals, offset=0):
    return (-b_svm - offset - w_svm[0] * x_vals) / w_svm[1]

ax.plot(xx_s, boundary_y(xx_s),       color=TEAL,  linewidth=2, label='Decision boundary')
ax.plot(xx_s, boundary_y(xx_s,  1),   color=TEAL,  linewidth=1, linestyle='--', label='Margin ($\\pm 1$)')
ax.plot(xx_s, boundary_y(xx_s, -1),   color=TEAL,  linewidth=1, linestyle='--')

ax.set_title(f'SVM primal — margin = {margin:.3f}', color='white')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

**What to notice:** only the points *on* the margin lines (functional margin `= 1`) are circled
— the **support vectors**. Their KKT multipliers `αᵢ > 0`; every other point has `αᵢ = 0` and could
be moved (or deleted) without changing the boundary. The SVM *is* a KKT problem: the support
vectors are exactly the active constraints.

## 4 — Dual Objective is Concave

For the 1-D case with two points $x_1=+1$ (class $+1$) and $x_2=-1$ (class $-1$),
the SVM dual objective in a single variable $\alpha$ (using symmetry $\alpha_1 = \alpha_2 = \alpha$,
$\sum \alpha_i y_i = 0$) is:

$$D(\alpha) = 2\alpha - \alpha^2 \cdot (1 \cdot 1 \cdot 1 + 1 \cdot 1 \cdot 1) / 2 = 2\alpha - \alpha^2$$

This is a downward-opening parabola — the dual is **concave**, as expected for the dual of a convex
primal problem. Maximising a concave function has the same guarantees as minimising a convex one.

In [ ]:
# Two-point 1-D SVM dual
# Data: x1=+1 (y=+1), x2=-1 (y=-1)
# Dual: D(alpha) = alpha1 + alpha2 - 0.5*(alpha1^2 * y1*y1*x1*x1
#                                        + 2*alpha1*alpha2*y1*y2*x1*x2
#                                        + alpha2^2 * y2*y2*x2*x2)
# With alpha1=alpha2=a and y1=y2=1, x1=1, x2=-1:
#   D(a) = 2a - 0.5*(a^2 - 2a^2 + a^2) = 2a - 0.5*0 = 2a  <-- degenerate 1-D case
# Use a richer 2-point example: y1=+1, y2=-1, x1=1, x2=-1
#   D(a1, a2) = a1 + a2 - 0.5*(a1^2*(+1)(+1)(1)(1) + 2*a1*a2*(+1)(-1)(1)(-1) + a2^2*(-1)(-1)(-1)(-1))
#             = a1 + a2 - 0.5*(a1^2 + 2*a1*a2 + a2^2)
# With constraint sum(a_i y_i)=0 => a1 - a2 = 0 => a1=a2=a:
#   D(a) = 2a - 0.5*(a^2 + 2a^2 + a^2) = 2a - 2a^2

alpha = np.linspace(0, 1.2, 300)
D = 2 * alpha - 2 * alpha**2   # concave dual objective
alpha_opt = 0.5
D_opt = 2 * alpha_opt - 2 * alpha_opt**2

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(alpha, D, color=BRAND, linewidth=2, label='$D(\\alpha) = 2\\alpha - 2\\alpha^2$')
ax.scatter([alpha_opt], [D_opt], color=TEAL, s=100, zorder=6,
           label=f'$\\alpha^* = {alpha_opt}$, $D^* = {D_opt}$')
ax.axvline(alpha_opt, color=TEAL, linewidth=1, linestyle='--', alpha=0.6)
ax.axhline(D_opt, color=TEAL, linewidth=1, linestyle='--', alpha=0.6)
ax.fill_between(alpha, D, where=(D >= D.min()), color=BRAND, alpha=0.1)
ax.set_xlabel(r'$\alpha$')
ax.set_ylabel('Dual objective $D(\\alpha)$')
ax.set_title('SVM dual is concave — maximisation has same guarantee as convex minimisation', color='white')
ax.legend()
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

print(f'Optimal alpha* = {alpha_opt},  D* = {D_opt}')
print(f'Recovered w* = alpha*_1 * y_1 * x_1 + alpha*_2 * y_2 * x_2')
print(f'           = {alpha_opt}*1*1 + {alpha_opt}*(-1)*(-1) = {alpha_opt + alpha_opt} (margin = {2/(alpha_opt + alpha_opt):.2f})')

**What to notice:** the SVM **dual** objective is a downward parabola — **concave** — so
maximizing it has the same one-global-optimum guarantee as minimizing a convex function. This is
why SVMs are solved via the dual: it's concave, and the solution reveals the support vectors and
recovers `w` directly.

## 5. Gotchas & tradeoffs

- **Only *active* constraints matter.** At the optimum, inactive constraints (`gᵢ < 0`) have
  `μᵢ = 0` and can be ignored; the solution is shaped only by the active ones (the support
  vectors, the binding budget).
- **Strong duality needs convexity.** For convex problems the dual optimum equals the primal
  (zero duality gap); for non-convex ones a **gap** can remain, so the dual only lower-bounds.
- **KKT are necessary, not always sufficient.** Without convexity (or a constraint
  qualification) a KKT point needn't be a global optimum.
- **Equality vs inequality.** Equality multipliers `λ` are free-signed; inequality multipliers
  `μ` must be `≥ 0` (dual feasibility).

In [ ]:
# Active vs inactive constraints at the LP optimum (3, 1)
x_s = np.array([3.0, 1.0])
g = np.array([x_s[0]+x_s[1]-4, x_s[0]-3, x_s[1]-3, -x_s[0], -x_s[1]])
print('constraint values g_i:', g)
print('active   (g == 0):', list(np.where(np.abs(g) < 1e-9)[0]), '-> carry nonzero mu')
print('inactive (g <  0):', list(np.where(g < -1e-9)[0]), '-> mu = 0 (complementary slackness)')

**What to notice:** two constraints are active (indices 0 and 1) and three are slack. Only the
active ones get nonzero multipliers — so the optimum is determined by a *handful* of binding
constraints, exactly why an SVM depends on only its support vectors, not the whole dataset.

## Key takeaways

- **Lagrange multipliers** (equality): the optimum is where `∇f = λ∇h` — parallel gradients,
  tangency.
- **KKT conditions** (inequality): stationarity + primal/dual feasibility + **complementary
  slackness**; only *active* constraints shape the solution.
- **SVM support vectors = active KKT constraints**; the **dual is concave**, giving convex-style
  guarantees.
- Use `scipy.optimize.minimize` / `cvxpy` / `sklearn` in practice; strong duality (zero gap)
  requires convexity.

**Next:** [Loss Functions](https://ml-viz-ruby.vercel.app/courses/optimization-ml/04-loss-functions).

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set,
and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently
when your answer is right.

### Exercise 1 — `lagrange_equality(f, grad_f, h, grad_h, x0, lr=0.01, n_steps=500)`

Implement **Lagrangian gradient ascent/descent** for equality-constrained problems:

$$\mathcal{L}(x, \lambda) = f(x) + \lambda h(x)$$

The update rules are:
- $x \leftarrow x - \eta \nabla_x \mathcal{L} = x - \eta(\nabla f(x) + \lambda \nabla h(x))$ — **descent** on $x$
- $\lambda \leftarrow \lambda + \eta h(x)$ — **ascent** on $\lambda$ (we want to maximise over $\lambda$)

Test on the worked example: minimise $x^2 + y^2$ s.t. $x + y = 1$.
The solution should converge to $(0.5, 0.5)$.

In [ ]:
def lagrange_equality(f, grad_f, h, grad_h, x0, lr=0.01, n_steps=500):
    """
    Gradient ascent/descent on L(x, lam) = f(x) + lam * h(x).

    Args:
        f:       objective function R^n -> R
        grad_f:  gradient of f, R^n -> R^n
        h:       constraint function R^n -> R  (should be 0 at optimum)
        grad_h:  gradient of h, R^n -> R^n
        x0:      initial point, array of shape (n,)
        lr:      learning rate
        n_steps: number of update steps

    Returns:
        (x, lam): final parameter vector and multiplier
    """
    x = np.array(x0, dtype=float)
    lam = 0.0

    for _ in range(n_steps):
        # TODO(you):
        # 1. Compute gradient of L w.r.t. x: grad_f(x) + lam * grad_h(x)
        # 2. Update x: x -= lr * grad_L_x
        # 3. Update lam: lam += lr * h(x)   (ascent on the dual variable)
        pass

    return x, lam

In [ ]:
# Test: min x^2 + y^2  s.t. x + y = 1  =>  solution (0.5, 0.5)
f_test     = lambda v: v[0]**2 + v[1]**2
grad_f_test = lambda v: np.array([2*v[0], 2*v[1]])
h_test     = lambda v: v[0] + v[1] - 1.0
grad_h_test = lambda v: np.array([1.0, 1.0])

x_star, lam_star = lagrange_equality(
    f_test, grad_f_test, h_test, grad_h_test,
    x0=np.array([0.0, 0.0]), lr=0.01, n_steps=1000
)

assert np.linalg.norm(x_star - np.array([0.5, 0.5])) < 0.05, \
    f'Expected x* ~ [0.5, 0.5], got {x_star}'
assert abs(h_test(x_star)) < 0.05, \
    f'Constraint should be ~0 at optimum, got {h_test(x_star):.4f}'
print('\u2705 Exercise 1 passed')

<details>
<summary>💡 Show solution</summary>

```python
def lagrange_equality(f, grad_f, h, grad_h, x0, lr=0.01, n_steps=500):
    x = np.array(x0, dtype=float)
    lam = 0.0
    for _ in range(n_steps):
        grad_L = grad_f(x) + lam * grad_h(x)
        x   -= lr * grad_L
        lam += lr * h(x)
    return x, lam
```

</details>

### Exercise 2 — `check_kkt(x_star, mu_star, g_funcs, grad_f, grad_g_funcs, tol=0.01)`

Implement a KKT checker for inequality-constrained problems (no equality constraints).
The four KKT conditions are:

1. **Stationarity:** $\|\nabla f(x^*) + \sum_i \mu_i^* \nabla g_i(x^*)\|_\infty < \text{tol}$
2. **Primal feasibility:** $g_i(x^*) \leq \text{tol}$ for all $i$
3. **Dual feasibility:** $\mu_i^* \geq -\text{tol}$ for all $i$
4. **Complementary slackness:** $|\mu_i^* g_i(x^*)| < \text{tol}$ for all $i$

Return a dict with keys `'stationarity'`, `'primal'`, `'dual'`, `'slackness'`.

In [ ]:
def check_kkt(x_star, mu_star, g_funcs, grad_f, grad_g_funcs, tol=0.01):
    """
    Verify KKT conditions for an inequality-constrained problem.

    Args:
        x_star:        optimal primal point, array shape (n,)
        mu_star:       dual variables, array shape (m,) with mu_i >= 0
        g_funcs:       list of m constraint functions g_i, each R^n -> R
        grad_f:        gradient of objective, R^n -> R^n
        grad_g_funcs:  list of m gradient functions for g_i, each R^n -> R^n
        tol:           numerical tolerance

    Returns:
        dict with keys 'stationarity', 'primal', 'dual', 'slackness' (all bool)
    """
    x_star  = np.asarray(x_star,  dtype=float)
    mu_star = np.asarray(mu_star, dtype=float)

    # TODO(you):
    # 1. Stationarity: compute grad_f(x*) + sum_i mu_i * grad_g_i(x*); check inf-norm < tol
    # 2. Primal feasibility: g_i(x*) <= tol for all i
    # 3. Dual feasibility: mu_i >= -tol for all i
    # 4. Complementary slackness: |mu_i * g_i(x*)| < tol for all i

    return {
        'stationarity': ...,
        'primal':       ...,
        'dual':         ...,
        'slackness':    ...,
    }

In [ ]:
# LP: min -2x - y  s.t. x+y<=4, x<=3, y<=3, x>=0, y>=0
# Optimal (3, 1), active: g1=x+y-4=0, g2=x-3=0; mu=[1,1,0,0,0]
x_lp  = np.array([3.0, 1.0])
mu_lp = np.array([1.0, 1.0, 0.0, 0.0, 0.0])

g_lp = [
    lambda v: v[0] + v[1] - 4,
    lambda v: v[0] - 3,
    lambda v: v[1] - 3,
    lambda v: -v[0],
    lambda v: -v[1],
]
grad_f_lp = lambda v: np.array([-2.0, -1.0])
grad_g_lp = [
    lambda v: np.array([1.0, 1.0]),
    lambda v: np.array([1.0, 0.0]),
    lambda v: np.array([0.0, 1.0]),
    lambda v: np.array([-1.0, 0.0]),
    lambda v: np.array([0.0, -1.0]),
]

result = check_kkt(x_lp, mu_lp, g_lp, grad_f_lp, grad_g_lp)
print('KKT check result:', result)

assert result['stationarity'], 'Stationarity should hold'
assert result['primal'],       'Primal feasibility should hold'
assert result['dual'],         'Dual feasibility should hold'
assert result['slackness'],    'Complementary slackness should hold'
print('\u2705 Exercise 2 passed')

<details>
<summary>💡 Show solution</summary>

```python
def check_kkt(x_star, mu_star, g_funcs, grad_f, grad_g_funcs, tol=0.01):
    x_star  = np.asarray(x_star,  dtype=float)
    mu_star = np.asarray(mu_star, dtype=float)

    # 1. Stationarity
    stationarity_vec = grad_f(x_star).copy()
    for mu_i, grad_g_i in zip(mu_star, grad_g_funcs):
        stationarity_vec += mu_i * grad_g_i(x_star)
    stationarity = bool(np.max(np.abs(stationarity_vec)) < tol)

    # 2. Primal feasibility
    g_vals = np.array([g(x_star) for g in g_funcs])
    primal = bool(np.all(g_vals <= tol))

    # 3. Dual feasibility
    dual = bool(np.all(mu_star >= -tol))

    # 4. Complementary slackness
    slackness = bool(np.all(np.abs(mu_star * g_vals) < tol))

    return {'stationarity': stationarity, 'primal': primal,
            'dual': dual, 'slackness': slackness}
```

</details>